# CropMedic Batch 1 — reusable crop trainer

Use this notebook after a crop dataset has been prepared and audited locally. The ZIP must contain the class folders at its root. Google Drive stores the ZIP, checkpoints and final artifacts so Colab disconnects can resume.

In [ ]:
# Change only this value for the crop you are training.
CROP_ID = 'soybean'  # examples: soybean, wheat, sorghum
HEAD_EPOCHS = 10
FINE_EPOCHS = 18
print('Crop:', CROP_ID)

In [ ]:
!nvidia-smi
%cd /content
!test -d CropMedic/.git && git -C CropMedic pull origin production-v1 || git clone -b production-v1 https://github.com/macdonaldchigutiro/CropMedic.git
%cd /content/CropMedic/crop_medic_backend/training
!pip -q install -r requirements-training.txt

In [ ]:
from google.colab import drive, files
from pathlib import Path
import shutil, zipfile
drive.mount('/content/drive')
root = Path('/content/drive/MyDrive/CropMedic')
dataset_dir = root / 'datasets'
checkpoint_dir = root / 'checkpoints' / CROP_ID
artifact_dir = root / 'batch1' / CROP_ID
dataset_dir.mkdir(parents=True, exist_ok=True)
checkpoint_dir.mkdir(parents=True, exist_ok=True)
artifact_dir.mkdir(parents=True, exist_ok=True)
drive_zip = dataset_dir / f'{CROP_ID}_for_colab.zip'
print('Persistent root:', root)

In [ ]:
if not drive_zip.exists():
    print('Upload the already-audited ZIP for', CROP_ID)
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise RuntimeError('Upload exactly one dataset ZIP.')
    zip_name = next(iter(uploaded))
    if not zip_name.lower().endswith('.zip'):
        raise RuntimeError('The uploaded file must be a ZIP archive.')
    shutil.copy2(zip_name, drive_zip)
else:
    print('Reusing Drive dataset:', drive_zip)

target = Path('data_batch1') / CROP_ID
if target.exists():
    shutil.rmtree(target)
target.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(drive_zip) as z:
    z.extractall(target)
print('Extracted:', target.resolve())

In [ ]:
!python audit_crop_dataset.py --data-dir data_batch1/$CROP_ID --crop-id $CROP_ID

**Stop if the audit does not say `Pilot audit ready: True`.** Do not train from uncertain labels, corrupt images or cross-label duplicates.

In [ ]:
!python train_crop_model.py --crop-id $CROP_ID --data-dir data_batch1/$CROP_ID --output-dir /content/drive/MyDrive/CropMedic/batch1/$CROP_ID --checkpoint-dir /content/drive/MyDrive/CropMedic/checkpoints/$CROP_ID --head-epochs $HEAD_EPOCHS --fine-epochs $FINE_EPOCHS

In [ ]:
import json
metrics_path = artifact_dir / 'metrics.json'
if not metrics_path.exists():
    raise RuntimeError('Training has not finished yet.')
m = json.loads(metrics_path.read_text())
print('TEST ACCURACY:', m['test_accuracy'])
print('MACRO F1:', m['macro_f1'])
print(json.dumps(m['per_class'], indent=2))

The model remains **pilot only** after this notebook. Freeze the metrics and run independent external validation/rejection testing before backend promotion.